# CogAttention — Visual Stroop (Multimodal)

**Track:** Attention — Selective Attention (Visual)
**Benchmark:** CogAttention v1.0
**Tasks:** visual_stroop

---

## Methodology

Tests whether vision-language models can separate low-level visual features
(ink color) from high-level semantic content (word meaning) using procedurally
generated Visual Stroop images.

The classic Stroop effect (Stroop, 1935) demonstrates that reading a color word
(e.g., "RED") interferes with naming the ink color when the two conflict
(e.g., the word "RED" printed in blue ink). This is one of the most robust
findings in cognitive psychology.

### Cognitive Science Grounding

- **Selective attention** (Cherry, 1953; Broadbent, 1958): filtering relevant from irrelevant stimuli
- **Stroop interference** (Stroop, 1935; MacLeod, 1991): automatic word reading competes with color naming
- **Inhibitory control** (Diamond, 2013): suppressing prepotent responses

### Difficulty Scaling

| Level    | Items | Font Size | Noise | Distractors |
|----------|-------|-----------|-------|-------------|
| Easy     | 1     | 60px      | No    | 0           |
| Medium   | 2     | 48px      | No    | 2           |
| Hard     | 3     | 36px      | Yes   | 4           |
| Expert   | 4     | 28px      | Yes   | 6           |
| Frontier | 5     | 24px      | Yes   | 8           |

No external datasets — all images generated with PIL at runtime. Zero data leakage.

### Scoring

SDK assertion pass rate = per-element accuracy. Each item checks whether the
response contains the correct ink color name (case-insensitive regex match).

---

`<!-- COGATTENTION-BENCH-CANARY-7E5E99CE7F54 -->`

In [ ]:
# ======================================================================
# Cell 1: Imports + Inline Helpers + Generator + Assertions
# CogAttention — Visual Stroop (Multimodal)
# ======================================================================

import kaggle_benchmarks as kbench

import json
import re
import random
import base64
import math
from io import BytesIO

try:
    from PIL import Image, ImageDraw, ImageFont
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

# ── Constants ──────────────────────────────────────────────────────────

ITEMS_PER_DIFFICULTY = 30
DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard", "Expert", "Frontier"]

COLOR_WORDS = ["RED", "BLUE", "GREEN", "YELLOW", "ORANGE", "PURPLE", "PINK", "BROWN"]
COLOR_HEX = {
    "RED": "#FF0000", "BLUE": "#0000FF", "GREEN": "#008000", "YELLOW": "#FFD700",
    "ORANGE": "#FF8C00", "PURPLE": "#800080", "PINK": "#FF69B4", "BROWN": "#8B4513",
}
COLOR_NAMES = list(COLOR_HEX.keys())

DIFFICULTY_CONFIG = {
    "Easy": {"n_items": 1, "font_size": 60, "bg_noise": False, "distractor_shapes": 0},
    "Medium": {"n_items": 2, "font_size": 48, "bg_noise": False, "distractor_shapes": 2},
    "Hard": {"n_items": 3, "font_size": 36, "bg_noise": True, "distractor_shapes": 4},
    "Expert": {"n_items": 4, "font_size": 28, "bg_noise": True, "distractor_shapes": 6},
    "Frontier": {"n_items": 5, "font_size": 24, "bg_noise": True, "distractor_shapes": 8},
}

# ── Image Generator ───────────────────────────────────────────────────

def _generate_stroop_image(
    word, ink_color, font_size, bg_noise, distractor_shapes, rng,
    img_width=500, img_height=300,
):
    """Generate a single Stroop image and return as base64 PNG string."""
    if not PIL_AVAILABLE:
        return f"[IMAGE_PLACEHOLDER: word={word} color={ink_color}]"

    bg_color = (240, 240, 240)
    img = Image.new("RGB", (img_width, img_height), color=bg_color)
    draw = ImageDraw.Draw(img)

    # Add distractor shapes
    for _ in range(distractor_shapes):
        shape_type = rng.choice(["rect", "ellipse"])
        x1 = rng.randint(0, img_width - 60)
        y1 = rng.randint(0, img_height - 40)
        x2 = x1 + rng.randint(20, 60)
        y2 = y1 + rng.randint(15, 40)
        shape_color = rng.choice(list(COLOR_HEX.values()))
        if shape_type == "rect":
            draw.rectangle([x1, y1, x2, y2], fill=shape_color, outline=None)
        else:
            draw.ellipse([x1, y1, x2, y2], fill=shape_color, outline=None)

    # Add background noise (random dots)
    if bg_noise:
        for _ in range(50):
            x = rng.randint(0, img_width - 1)
            y = rng.randint(0, img_height - 1)
            noise_color = (rng.randint(180, 230), rng.randint(180, 230), rng.randint(180, 230))
            draw.point((x, y), fill=noise_color)

    # Draw the Stroop word in conflicting ink color
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", font_size)
    except (IOError, OSError):
        font = ImageFont.load_default()

    ink_hex = COLOR_HEX[ink_color]
    bbox = draw.textbbox((0, 0), word, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x = (img_width - text_w) // 2
    y = (img_height - text_h) // 2

    draw.text((x, y), word, fill=ink_hex, font=font)

    # Convert to base64
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode("ascii")
    return img_b64


def _generate_visual_stroop_instance(instance_idx, difficulty, seed=2026):
    """Generate a single Visual Stroop instance as a dict."""
    rng = random.Random(seed + instance_idx * 1031 + DIFFICULTY_LEVELS.index(difficulty) * 4201)
    config = DIFFICULTY_CONFIG[difficulty]

    items = []
    gold_answers = {}
    images_b64 = []

    for i in range(config["n_items"]):
        word = rng.choice(COLOR_WORDS)
        # Ensure ink color conflicts with word
        ink_color = rng.choice([c for c in COLOR_NAMES if c != word])

        img_b64 = _generate_stroop_image(
            word=word,
            ink_color=ink_color,
            font_size=config["font_size"],
            bg_noise=config["bg_noise"],
            distractor_shapes=config["distractor_shapes"],
            rng=rng,
        )

        items.append({"word": word, "ink_color": ink_color.lower()})
        gold_answers[str(i + 1)] = ink_color.lower()
        images_b64.append(img_b64)

    # Build prompt
    if config["n_items"] == 1:
        prompt = (
            "You will see an image containing a color word printed in a specific ink color. "
            "The word may NOT match the ink color. "
            "What is the PHYSICAL INK COLOR of the text? Do NOT read the word \u2014 only state the "
            "color of the pixels/ink.\n\n"
            "ANSWER: [color name]"
        )
    else:
        prompt = (
            f"You will see {config['n_items']} images, each containing a color word printed "
            "in a specific ink color. The word may NOT match the ink color. "
            "For each image, state the PHYSICAL INK COLOR of the text. "
            "Do NOT read the word \u2014 only state the color of the pixels/ink.\n\n"
            "ANSWER:\n" + "\n".join(f"{i+1}. [color name]" for i in range(config["n_items"]))
        )

    task_id = f"visual_stroop_{difficulty.lower()}_{instance_idx:03d}"

    # Return gold as string for single item, dict for multi-item
    gold = gold_answers["1"] if config["n_items"] == 1 else gold_answers

    return {
        "task_id": task_id,
        "task_type": "visual_stroop",
        "difficulty": difficulty,
        "prompt": prompt,
        "gold": gold,
        "images": images_b64,
        "items": items,
    }


def generate_visual_stroop_dataset(seed=2026):
    """Generate full Visual Stroop dataset across all difficulty tiers."""
    dataset = []
    idx = 0
    for diff in DIFFICULTY_LEVELS:
        for i in range(ITEMS_PER_DIFFICULTY):
            inst = _generate_visual_stroop_instance(idx, diff, seed)
            dataset.append(inst)
            idx += 1
    return dataset


# ── Assertion Function ─────────────────────────────────────────────────

def run_assertions_visual_stroop(response, gold, kbench):
    # Color variant mapping — accept partial/variant matches
    COLOR_VARIANTS = {
        "red": ["red", "reddish", "crimson", "scarlet", "vermilion"],
        "blue": ["blue", "bluish", "navy", "cobalt", "azure", "indigo"],
        "green": ["green", "greenish", "lime", "emerald", "olive"],
        "yellow": ["yellow", "yellowish", "gold", "golden", "amber"],
        "orange": ["orange", "orangish", "tangerine", "amber"],
        "purple": ["purple", "purplish", "violet", "magenta", "plum", "lavender"],
        "pink": ["pink", "pinkish", "magenta", "rose", "fuchsia"],
        "brown": ["brown", "brownish", "tan", "maroon", "chocolate", "dark red"],
    }
    def _color_matches(gold_color, response_text):
        variants = COLOR_VARIANTS.get(gold_color, [gold_color])
        for v in variants:
            if re.search(rf"(?i)\b{re.escape(v)}\b", response_text):
                return True
        return False

    if isinstance(gold, str):
        result_str = "PASSED" if _color_matches(gold, response) else "FAILED"
        kbench.assertions.assert_contains_regex(
            r"PASSED", result_str,
            expectation=f"Response should contain ink color '{gold}'"
        )
    else:
        n = len(gold)
        min_required = max(1, n - 1)
        correct = 0
        for idx, color in gold.items():
            # Check positional match first
            pattern = rf"(?i)\b{re.escape(idx)}\s*[.):\-]\s*.*"
            match = re.search(pattern, response)
            if match and _color_matches(color, match.group(0)):
                correct += 1
            elif _color_matches(color, response):
                # Fallback: color appears anywhere
                correct += 1
        result_str = "PASSED" if correct >= min_required else f"FAILED_{correct}_of_{n}"
        kbench.assertions.assert_contains_regex(
            r"PASSED", result_str,
            expectation=f"Should identify at least {min_required}/{n} ink colors correctly (got {correct})"
        )

print("Visual Stroop helpers loaded.")

In [ ]:
# ======================================================================
# Cell 2: Task Definition + Dataset Generation
# ======================================================================

@kbench.task(name="cogattention_visual_stroop")
def cogattention_visual_stroop(llm, prompt: str, gold_json: str, image_data: str, task_id: str, difficulty: str):
    """CogAttention visual stroop task."""
    images = json.loads(image_data)
    # Multimodal prompt with base64 images
    full_prompt = prompt
    for i, img_b64 in enumerate(images):
        full_prompt += f"\n\n[Image {i+1} (base64 PNG)]: {img_b64}"
    response = llm.prompt(full_prompt)
    gold = json.loads(gold_json)
    run_assertions_visual_stroop(response, gold, kbench)

# Generate dataset at runtime (keeps notebook <1MB)
print("Generating Visual Stroop dataset...")
raw_dataset = generate_visual_stroop_dataset(seed=2026)
DATASET = []
for inst in raw_dataset:
    DATASET.append({
        "task_id": inst["task_id"],
        "task_type": inst["task_type"],
        "difficulty": inst["difficulty"],
        "prompt": inst["prompt"],
        "gold_json": json.dumps(inst["gold"]),
        "image_data": json.dumps(inst["images"]),
    })
print(f"Generated {len(DATASET)} items")

In [ ]:
# ======================================================================
# Cell 3: Execution Loop
# ======================================================================

TASK_DISPATCH = {"visual_stroop": cogattention_visual_stroop}
n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        image_data=item["image_data"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )
print(f"\nCompleted {n_total} items for Visual Stroop")